# FinGraph AI — 01 Exploratory Data Analysis (EDA)

**Svrha:** detaljno istražiti Elliptic++ transakcijski graf prije optimizacije modela.

Ovaj notebook je nastavak `00_colab_quickstart.ipynb`. Ne trenira modele i ne mijenja postojeće rezultate. Učitava originalne podatke preko projektnog loadera i čita već sačuvane metrike/predikcije XGBoost i GraphSAGE modela.

## Šta ćemo analizirati?
1. Strukturu i raspodjelu oznaka.
2. Promjene kroz 49 vremenskih koraka.
3. Nedostajuće vrijednosti i osnovne statistike karakteristika.
4. Strukturu grafa i stepene čvorova.
5. Vremenske odnose i oznake na krajevima veza.
6. Već dobijene rezultate modela.

**Metodološka napomena:** `illicit` i `licit` su poznate oznake iz dataseta. `unknown` nije negativna klasa i ne smije se tretirati kao legitimna transakcija. Analiza mrežnih obrazaca ne predstavlja dokaz kriminalne aktivnosti.

## 0. Priprema okruženja

Pokreni ćelije redom. Ako koristiš isti Google Drive folder kao u `00_colab_quickstart.ipynb`, putanja ispod treba odgovarati tvom okruženju.

Ako se projekt nalazi na drugoj lokaciji, promijeni samo `PROJECT_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

PROJECT_DIR = Path("/content/drive/MyDrive/fingraph-ai/FinGraph_AI_MPVI_Starter")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Projekt nije pronađen: {PROJECT_DIR}. Provjeri PROJECT_DIR."
    )

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

print("Radni direktorij:", Path.cwd())
print("Dataset folder:", (PROJECT_DIR / "data/raw").resolve())
print("Postojeći output folder:", (PROJECT_DIR / "outputs").exists())

In [ ]:
# EDA ne zahtijeva PyTorch Geometric; dovoljne su osnovne projektne zavisnosti.
%pip install -q -e .

import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fingraph.data import load_elliptic
from fingraph.splits import make_time_splits
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("outputs/eda")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = {1: "Illicit", 0: "Licit", -1: "Unknown"}

print("Osnovne biblioteke uspješno učitane.")

## 1. Učitavanje podataka i osnovna provjera

Koristimo isti `load_elliptic()` loader kao u treniranju. Time osiguravamo da EDA koristi isto poravnanje transakcijskih ID-jeva, oznaka i veza kao naši modeli.

Interno kodiranje oznaka u projektu:
- `1` = illicit
- `0` = licit
- `-1` = unknown

In [ ]:
DATA_DIR = Path("data/raw")
graph = load_elliptic(DATA_DIR)
splits = make_time_splits(graph)

src, dst = graph.edge_index
n_nodes = graph.num_nodes
n_edges = graph.num_edges
n_features = graph.features.shape[1]

overview = pd.DataFrame({
    "Pokazatelj": [
        "Broj transakcija / čvorova",
        "Broj usmjerenih veza",
        "Broj ulaznih karakteristika",
        "Broj vremenskih koraka",
        "Broj nedostajućih vrijednosti karakteristika",
        "Broj izolovanih čvorova (bez ulaznih i izlaznih veza)",
    ],
    "Vrijednost": [
        n_nodes,
        n_edges,
        n_features,
        len(np.unique(graph.times)),
        int(np.isnan(graph.features).sum()),
        int(((np.bincount(src, minlength=n_nodes) + np.bincount(dst, minlength=n_nodes)) == 0).sum()),
    ],
})

display(overview)
print("Prvih 15 naziva karakteristika:")
print(graph.feature_names[:15])

In [ ]:
feature_inventory = pd.DataFrame({
    "feature_index": np.arange(len(graph.feature_names)),
    "feature_name": list(graph.feature_names),
})
display(feature_inventory)
feature_inventory.to_csv(TABLE_DIR / "feature_inventory.csv", index=False)

### Kako tumačiti ovaj pregled?

- **Čvor** predstavlja jednu transakciju.
- **Usmjerena veza** predstavlja relaciju između dvije transakcije prema `txs_edgelist.csv`.
- **Karakteristike** predstavljaju numeričke informacije koje model koristi.
- **Izolovan čvor** nema evidentiranu vezu u posmatranom grafu.

Broj izolovanih čvorova nije automatski problem: transakcija može imati karakteristike i oznaku, a da u ovom konkretnom grafu nema evidentiranu vezu.

**Napomena o broju karakteristika:** službeni opis Elliptic++ navodi 183 features. U našem loaderu se `Time step` čuva kao zasebna vremenska varijabla, a `txId` i oznaka klase se ne koriste kao prediktori. Zato audit prikazuje 182 ulazne numeričke karakteristike modela.

## 2. Raspodjela klasa

Prvo provjeravamo koliko imamo transakcija u svakoj klasi. Prikazat ćemo i broj svih transakcija i raspodjelu samo među poznatim oznakama.

Ovo je važno jer se model trenira i evaluira na poznatim oznakama, dok `unknown` čvorovi mogu ostati dio mrežnog konteksta.

In [ ]:
label_counts = (
    pd.Series(graph.labels)
    .map(LABEL_NAMES)
    .value_counts()
    .reindex(["Illicit", "Licit", "Unknown"], fill_value=0)
)

label_summary = pd.DataFrame({
    "Broj transakcija": label_counts,
    "Udio svih transakcija (%)": (label_counts / n_nodes * 100).round(3),
})
display(label_summary)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(label_summary.index, label_summary["Broj transakcija"])
ax.set_title("Raspodjela oznaka u Elliptic++")
ax.set_ylabel("Broj transakcija")
ax.set_xlabel("Klasa")
ax.ticklabel_format(axis="y", style="plain")
for bar, value in zip(bars, label_summary["Broj transakcija"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(value):,}",
            ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_class_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

known_total = int((graph.labels >= 0).sum())
positive_total = int((graph.labels == 1).sum())
print(f"Poznato označene transakcije: {known_total:,}")
print(f"Illicit udio među poznatim oznakama: {positive_total / known_total:.2%}")

### Tumačenje

Očekujemo izraženu neravnotežu klasa: legitimnih transakcija ima mnogo više nego nezakonitih, a najveći dio skupa ima oznaku `unknown`.

Zbog toga:
- obična accuracy metrika nije dovoljna;
- PR-AUC je važna primarna metrika;
- nepoznate oznake ne uključujemo u izračun nadziranih metrika;
- rezultate uvijek tumačimo uz broj pozitivnih primjera.

## 3. Promjene kroz vremenske korake

Elliptic++ ima 49 vremenskih koraka. Provjeravamo da li se broj transakcija, udio poznatih oznaka i udio `illicit` transakcija mijenjaju kroz vrijeme.

Ova analiza je važna za razumijevanje razlike između validacijskih koraka 35–41 i testnih koraka 42–49.

In [ ]:
time_df = pd.DataFrame({
    "time_step": graph.times,
    "label": pd.Series(graph.labels).map(LABEL_NAMES).to_numpy(),
})

time_counts = (
    time_df.groupby(["time_step", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Illicit", "Licit", "Unknown"], fill_value=0)
    .sort_index()
)

time_counts["Known"] = time_counts["Illicit"] + time_counts["Licit"]
time_counts["Illicit_share_known"] = (
    time_counts["Illicit"] / time_counts["Known"].replace(0, np.nan)
)
display(time_counts.head(10))

time_counts.to_csv(TABLE_DIR / "class_distribution_by_time.csv")

fig, ax = plt.subplots(figsize=(13, 5))
ax.stackplot(
    time_counts.index,
    time_counts["Illicit"],
    time_counts["Licit"],
    time_counts["Unknown"],
    labels=["Illicit", "Licit", "Unknown"],
    alpha=0.85,
)
ax.axvline(34.5, linestyle="--", linewidth=1.5, label="Granica treninga")
ax.axvline(41.5, linestyle=":", linewidth=1.8, label="Granica validacije")
ax.set_title("Broj transakcija po klasi i vremenskom koraku")
ax.set_xlabel("Vremenski korak")
ax.set_ylabel("Broj transakcija")
ax.legend(loc="upper left", ncol=2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_class_counts_by_time.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(
    time_counts.index,
    time_counts["Illicit_share_known"] * 100,
    marker="o",
    markersize=3,
)
ax.axvline(34.5, linestyle="--", linewidth=1.5, label="Granica treninga")
ax.axvline(41.5, linestyle=":", linewidth=1.8, label="Granica validacije")
ax.set_title("Udio illicit transakcija među poznato označenim transakcijama")
ax.set_xlabel("Vremenski korak")
ax.set_ylabel("Illicit udio (%)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_illicit_share_by_time.png", dpi=180, bbox_inches="tight")
plt.show()

split_rows = []
for split_name, mask in [
    ("Train", splits.train),
    ("Validation", splits.valid),
    ("Test", splits.test),
]:
    y_split = graph.labels[mask]
    split_rows.append({
        "Skup": split_name,
        "Broj poznato označenih": len(y_split),
        "Illicit": int((y_split == 1).sum()),
        "Licit": int((y_split == 0).sum()),
        "Illicit udio (%)": round(float((y_split == 1).mean() * 100), 3),
    })

split_summary = pd.DataFrame(split_rows)
display(split_summary)
split_summary.to_csv(TABLE_DIR / "split_class_summary.csv", index=False)

### Šta posebno posmatrati?

1. Da li postoje vremenski koraci s naglim porastom `illicit` oznaka?
2. Da li se udio `illicit` transakcija smanjuje ili povećava?
3. Da li su validacija i test po raspodjeli klasa slični treningu?
4. Postoje li koraci s vrlo malim brojem poznatih oznaka?

Ako se raspodjela mijenja, to može doprinijeti slabijim rezultatima na kasnijem testnom periodu. Sama promjena udjela klase ne dokazuje uzrok pada performansi; potrebno je posmatrati i karakteristike i mrežnu strukturu.

## 4. Nedostajuće vrijednosti i karakteristike

Prvo mjerimo koliko nedostajućih vrijednosti ima svaka karakteristika. Zatim računamo osnovne statistike za svaku kolonu.

Ovdje ne mijenjamo podatke. Imputacija se i dalje radi u modelima i parametri imputacije se uče samo na istorijskom trening periodu.

In [ ]:
feature_matrix = graph.features
missing_counts = np.isnan(feature_matrix).sum(axis=0)
missing_rates = missing_counts / n_nodes * 100

missing_df = pd.DataFrame({
    "feature": graph.feature_names,
    "missing_count": missing_counts,
    "missing_percent": missing_rates,
}).sort_values("missing_percent", ascending=False)

display(missing_df.head(20))
missing_df.to_csv(TABLE_DIR / "feature_missingness.csv", index=False)

top_missing = missing_df.head(20).sort_values("missing_percent")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_missing["feature"].astype(str), top_missing["missing_percent"])
ax.set_title("20 karakteristika s najvećim udjelom nedostajućih vrijednosti")
ax.set_xlabel("Nedostajuće vrijednosti (%)")
ax.set_ylabel("Karakteristika")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_top_feature_missingness.png", dpi=180, bbox_inches="tight")
plt.show()

stats_rows = []
for j, name in enumerate(graph.feature_names):
    col = feature_matrix[:, j]
    valid = col[np.isfinite(col)]
    if valid.size == 0:
        stats_rows.append({
            "feature": name, "valid_count": 0, "mean": np.nan, "std": np.nan,
            "min": np.nan, "median": np.nan, "max": np.nan, "variance": np.nan
        })
    else:
        stats_rows.append({
            "feature": name,
            "valid_count": int(valid.size),
            "mean": float(np.mean(valid)),
            "std": float(np.std(valid)),
            "min": float(np.min(valid)),
            "median": float(np.median(valid)),
            "max": float(np.max(valid)),
            "variance": float(np.var(valid)),
        })

feature_stats = pd.DataFrame(stats_rows)
feature_stats.to_csv(TABLE_DIR / "feature_summary.csv", index=False)

print("Karakteristika bez ijedne validne vrijednosti:",
      int((feature_stats["valid_count"] == 0).sum()))
print("Konstantne karakteristike:",
      int((feature_stats["variance"].fillna(0) == 0).sum()))
display(feature_stats.sort_values("variance", ascending=False).head(15))

### Tumačenje nedostajućih vrijednosti

Nedostajuća vrijednost nije isto što i nula. Nula može biti validna vrijednost neke karakteristike, dok `NaN` znači da vrijednost nije dostupna.

Zato u ovom projektu koristimo imputaciju medijanom. Medijani se računaju na trening periodu, a zatim se ista transformacija primjenjuje na validaciju i test.

U izvještaju ćemo navesti:
- ukupan broj nedostajućih vrijednosti;
- karakteristike s najviše nedostajućih vrijednosti;
- da li postoje potpuno prazne ili konstantne karakteristike.

## 5. Struktura grafa: stepeni čvorova

Za svaki čvor računamo:
- **in-degree:** broj ulaznih veza;
- **out-degree:** broj izlaznih veza;
- **total degree:** zbir ulaznih i izlaznih veza.

Ovo su osnovne topološke karakteristike. One opisuju koliko je transakcija povezana s ostatkom grafa, ali same po sebi ne označavaju transakciju kao sumnjivu.

In [ ]:
in_degree = np.bincount(dst, minlength=n_nodes)
out_degree = np.bincount(src, minlength=n_nodes)
total_degree = in_degree + out_degree

degree_df = pd.DataFrame({
    "txId": graph.tx_ids,
    "time_step": graph.times,
    "label": pd.Series(graph.labels).map(LABEL_NAMES).to_numpy(),
    "in_degree": in_degree,
    "out_degree": out_degree,
    "total_degree": total_degree,
})

degree_summary = degree_df[["in_degree", "out_degree", "total_degree"]].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T
display(degree_summary)

degree_df.groupby("label")[["in_degree", "out_degree", "total_degree"]].agg(
    ["count", "median", "mean", "max"]
)

In [ ]:
nonzero_degree = total_degree[total_degree > 0]
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.logspace(0, np.log10(max(2, int(nonzero_degree.max()) + 1)), 40)
ax.hist(nonzero_degree, bins=bins)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Raspodjela ukupnog stepena čvorova (log-log prikaz)")
ax.set_xlabel("Ukupni stepen (in-degree + out-degree)")
ax.set_ylabel("Broj čvorova")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_degree_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

known_degree = degree_df[degree_df["label"].isin(["Illicit", "Licit"])].copy()
known_degree["log1p_total_degree"] = np.log1p(known_degree["total_degree"])

fig, ax = plt.subplots(figsize=(8, 5))
groups = [
    known_degree.loc[known_degree["label"] == "Licit", "log1p_total_degree"],
    known_degree.loc[known_degree["label"] == "Illicit", "log1p_total_degree"],
]
ax.boxplot(groups, tick_labels=["Licit", "Illicit"], showfliers=False)
ax.set_title("Stepen čvorova po poznatoj klasi")
ax.set_ylabel("log(1 + ukupni stepen)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_degree_by_class.png", dpi=180, bbox_inches="tight")
plt.show()

degree_df.to_csv(TABLE_DIR / "node_degrees.csv", index=False)

### Kako tumačiti stepene?

Ako `illicit` čvorovi imaju viši prosječni stepen, to može ukazivati na drugačiju povezanost u mreži. Međutim, poređenje treba raditi i preko medijana i raspodjele, jer mali broj čvorova s ekstremno visokim stepenom može značajno povećati prosjek.

Ako se raspodjele mnogo preklapaju, sam stepen čvora nije dovoljan za pouzdanu klasifikaciju.

## 6. Vremenski odnosi između povezanih transakcija

Za svaku usmjerenu vezu `txId1 → txId2` računamo:

`time_delta = time(txId2) - time(txId1)`

To nam pokazuje da li veza ide od ranijeg ka kasnijem vremenskom koraku, unutar istog koraka ili od kasnijeg ka ranijem.

Ovo je deskriptivna analiza. Ne mijenjamo značenje originalne veze i ne zaključujemo da je svaka veza direktan transfer novca.

In [ ]:
edge_delta = graph.times[dst] - graph.times[src]

edge_time_summary = pd.DataFrame({
    "odnos": ["Izvor ranije od odredišta", "Isti vremenski korak", "Izvor kasnije od odredišta"],
    "broj_veza": [
        int((edge_delta > 0).sum()),
        int((edge_delta == 0).sum()),
        int((edge_delta < 0).sum()),
    ],
})
edge_time_summary["udio_posto"] = (
    edge_time_summary["broj_veza"] / len(edge_delta) * 100
).round(3)

display(edge_time_summary)
edge_time_summary.to_csv(TABLE_DIR / "edge_time_delta_summary.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(edge_delta, bins=np.arange(edge_delta.min() - 0.5, edge_delta.max() + 1.5, 1))
ax.set_title("Razlika vremenskih koraka na usmjerenim vezama")
ax.set_xlabel("time(txId2) - time(txId1)")
ax.set_ylabel("Broj veza")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_edge_time_delta.png", dpi=180, bbox_inches="tight")
plt.show()

### Zašto je ova analiza bitna?

U GraphSAGE implementaciji veze se tokom propagacije poruka pretvaraju u dvosmjerne, ali se prije toga filtriraju tako da oba kraja veze moraju biti opažena do evaluiranog vremenskog koraka.

Ova analiza nam pomaže da razumijemo koliko originalni graf sadrži vremenski naprijed, vremenski nazad i istovremeno povezane transakcije. U narednoj fazi možemo ispitati da li očuvanje smjera veze poboljšava rezultate.

## 7. Odnos oznaka na krajevima veza

Posmatramo samo veze čiji izvor i odredište imaju poznatu oznaku. `Unknown` čvorove ne uključujemo u ovu matricu, jer njihova klasa nije poznata.

Tabela prikazuje koliko često je izvorna klasa povezana s odredišnom klasom. Ovo je opis mreže, ne dokaz uzročnosti.

In [ ]:
known_edge_mask = (graph.labels[src] >= 0) & (graph.labels[dst] >= 0)

source_labels = pd.Series(graph.labels[src[known_edge_mask]]).map({0: "Licit", 1: "Illicit"})
target_labels = pd.Series(graph.labels[dst[known_edge_mask]]).map({0: "Licit", 1: "Illicit"})

edge_mixing = pd.crosstab(
    source_labels,
    target_labels,
    rownames=["Izvorna klasa"],
    colnames=["Odredišna klasa"],
).reindex(index=["Licit", "Illicit"], columns=["Licit", "Illicit"], fill_value=0)

display(edge_mixing)
edge_mixing.to_csv(TABLE_DIR / "edge_label_mixing_counts.csv")

row_normalized = edge_mixing.div(edge_mixing.sum(axis=1).replace(0, np.nan), axis=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(row_normalized.fillna(0).values, vmin=0, vmax=1)
ax.set_xticks([0, 1], labels=row_normalized.columns)
ax.set_yticks([0, 1], labels=row_normalized.index)
ax.set_xlabel("Odredišna klasa")
ax.set_ylabel("Izvorna klasa")
ax.set_title("Udio odredišnih klasa po izvornoj klasi")
for i in range(2):
    for j in range(2):
        value = row_normalized.iloc[i, j]
        ax.text(j, i, f"{value:.1%}" if pd.notna(value) else "N/A",
                ha="center", va="center")
fig.colorbar(im, ax=ax, label="Udio unutar izvorne klase")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_edge_label_mixing.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Poređenje postojećih modela

Ovdje ne treniramo ponovo. Učitavamo već spremljene `metrics.json` i `test_predictions.csv` datoteke iz početnog eksperimenta.

Provjeravamo:
- PR-AUC i ROC-AUC;
- precision, recall i F1;
- PR i ROC krivulje;
- matrice zabune pri pragu odabranom na validacijskom skupu;
- Precision@100.

Napomena: PR krivulja i ROC krivulja računaju se samo na poznato označenim testnim transakcijama koje se nalaze u spremljenim predikcijama.

In [ ]:
MODEL_DIRS = {
    "XGBoost": Path("outputs/xgboost"),
    "GraphSAGE": Path("outputs/graphsage"),
}

model_metrics = {}
model_predictions = {}

for model_name, folder in MODEL_DIRS.items():
    metrics_path = folder / "metrics.json"
    predictions_path = folder / "test_predictions.csv"

    if not metrics_path.exists() or not predictions_path.exists():
        raise FileNotFoundError(
            f"Nedostaje {metrics_path} ili {predictions_path}. "
            "Prvo pokreni 00_colab_quickstart.ipynb."
        )

    model_metrics[model_name] = json.loads(metrics_path.read_text(encoding="utf-8"))
    model_predictions[model_name] = pd.read_csv(predictions_path)

comparison_rows = []
for model_name, result in model_metrics.items():
    test = result["test"]
    comparison_rows.append({
        "Model": model_name,
        "PR-AUC": test["pr_auc"],
        "ROC-AUC": test["roc_auc"],
        "Precision": test["precision"],
        "Recall": test["recall"],
        "F1": test["f1"],
        "Precision@100": test["precision_at_100_or_less"],
        "Recall@100": test["recall_at_100_or_less"],
        "Threshold": test["threshold"],
    })

comparison = pd.DataFrame(comparison_rows).set_index("Model")
display(comparison.round(4))
comparison.to_csv(TABLE_DIR / "model_comparison.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_name, pred in model_predictions.items():
    y_true = pred["label"].to_numpy(dtype=int)
    y_score = pred["probability_illicit"].to_numpy(dtype=float)
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    ax.plot(recall, precision, label=f"{model_name} (AP={ap:.3f})")

ax.set_title("Precision–Recall krivulje na testnom skupu")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "09_precision_recall_curves.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for model_name, pred in model_predictions.items():
    y_true = pred["label"].to_numpy(dtype=int)
    y_score = pred["probability_illicit"].to_numpy(dtype=float)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", label="Nasumično rangiranje")
ax.set_title("ROC krivulje na testnom skupu")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "10_roc_curves.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
for model_name, pred in model_predictions.items():
    y_true = pred["label"].to_numpy(dtype=int)
    y_score = pred["probability_illicit"].to_numpy(dtype=float)
    threshold = float(model_metrics[model_name]["test"]["threshold"])
    y_pred = (y_score >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Licit", "Illicit"],
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, values_format=",", colorbar=False)
    ax.set_title(f"Matrica zabune — {model_name} (threshold={threshold:.2f})")
    fig.tight_layout()
    safe_name = model_name.lower().replace(" ", "_")
    fig.savefig(FIGURE_DIR / f"11_confusion_matrix_{safe_name}.png",
                dpi=180, bbox_inches="tight")
    plt.show()

### Kako tumačiti poređenje?

- **PR-AUC** poredi kvalitet rangiranja pozitivnih transakcija kroz različite pragove.
- **ROC-AUC** mjeri sposobnost rangiranja pozitivnih iznad negativnih primjera, ali može djelovati optimistično kod jake neravnoteže klasa.
- **Precision** govori koliki dio označenih transakcija zaista ima `illicit` oznaku.
- **Recall** govori koliki dio svih poznato označenih `illicit` transakcija je pronađen.
- **F1** predstavlja harmonijsku sredinu precision i recall pri odabranom pragu.
- **Precision@100** govori koliko je pozitivnih među 100 transakcija s najvećim skorom (ili među svim ako ih je manje od 100).

Pragovi se preuzimaju iz postojećih rezultata i odabrani su na validacijskom skupu. Ne biramo novi prag na testnom skupu.

## 9. Sažetak i automatsko spremanje EDA artefakata

Sve tabele i grafikoni se spremaju u `outputs/eda/`. Ovaj folder je isključen iz Git-a prema trenutnom `.gitignore`, što je korisno za velike ili generisane rezultate.

Na kraju provjerimo šta je napravljeno.

In [ ]:
summary = {
    "dataset": "Elliptic++ Transactions",
    "nodes": int(graph.num_nodes),
    "directed_edges": int(graph.num_edges),
    "features": int(graph.features.shape[1]),
    "time_steps": int(len(np.unique(graph.times))),
    "missing_feature_values": int(np.isnan(graph.features).sum()),
    "isolated_nodes": int(((in_degree + out_degree) == 0).sum()),
    "self_loops": int((src == dst).sum()),
    "unique_directed_edges": int(np.unique(np.column_stack([src, dst]), axis=0).shape[0]),
    "train_time_steps": [1, splits.train_end],
    "validation_time_steps": [splits.train_end + 1, splits.valid_end],
    "test_time_steps": [splits.valid_end + 1, int(graph.times.max())],
    "model_test_metrics": {
        name: model_metrics[name]["test"] for name in model_metrics
    },
}

(OUTPUT_DIR / "eda_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

print("EDA završena.")
print("Sažetak:", OUTPUT_DIR / "eda_summary.json")
print("Tabele:", TABLE_DIR.resolve())
print("Grafikoni:", FIGURE_DIR.resolve())

print("\nGenerisani fajlovi:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print("-", path)

## 10. Šta treba zaključiti nakon pokretanja?

Nemoj samo preuzeti grafikone. Za izvještaj zapiši odgovore na sljedeća pitanja:

1. Koliko je klasa neuravnotežena i zašto `unknown` nije negativna klasa?
2. Kako se udio `illicit` transakcija mijenja kroz 49 vremenskih koraka?
3. Da li postoje karakteristike s velikim udjelom nedostajućih vrijednosti?
4. Kakva je raspodjela stepena čvorova? Postoje li čvorovi s izrazito mnogo veza?
5. Da li se stepeni `illicit` i `licit` čvorova razlikuju?
6. Kakav je odnos vremenskih koraka na krajevima usmjerenih veza?
7. Da li se `illicit` čvorovi češće povezuju s drugim `illicit` čvorovima ili s `licit` čvorovima?
8. Šta PR i ROC krivulje govore o razlici XGBoost i GraphSAGE modela?

**Važno:** ne zaključuj da određena karakteristika uzrokuje nezakonitu aktivnost. EDA pokazuje povezanosti i raspodjele, ne uzročnost.

### Sljedeća faza nakon EDA

Na osnovu rezultata ovog notebooka uradit ćemo:
- ablacijski eksperiment sa lokalnim karakteristikama;
- poređenje XGBoost, MLP i GraphSAGE modela pod uporedivim ulazima;
- optimizaciju GraphSAGE hiperparametara;
- izdvajanje i vizualizaciju povezanih visokorizičnih podgrafova.